In [1]:
%pip install transformers 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\afagn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from transformers import pipeline

fichier_entree = "D:/XoNet/data/fongbe_french_corpus_final.csv"
fichier_sortie = "D:/XoNet/data/base.csv"

print("Chargement du CSV...")
df = pd.read_csv(fichier_entree, sep=',')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)


Chargement du CSV...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
print("Colonnes disponibles :", df.columns.tolist())

Colonnes disponibles : ['fon', 'fr']


In [9]:
def convertir_en_label(resultat, seuil=0.7):
    label = resultat['label'].upper()
    score = resultat['score']

    score_signe = score if label == "POSITIVE" else -score

    if score_signe > seuil:
        return 2  # Positif
    elif score_signe < -seuil:
        return 1  # Négatif
    else:
        return 0  # Neutre

print("Analyse des phrases en cours...")
resultats = classifier(df['fr'].tolist(), truncation=True)

labels_finaux = [convertir_en_label(r) for r in resultats]

df['sentiment_base'] = labels_finaux
df.to_csv(fichier_sortie, sep=',', index=False)

print(f"Terminé ! Fichier sauvegardé sous le nom : {fichier_sortie}")

# Vérifie la distribution obtenue
print(df['sentiment_base'].value_counts())

Analyse des phrases en cours...
Terminé ! Fichier sauvegardé sous le nom : D:/XoNet/data/base.csv
sentiment_base
0    48200
1    20119
2    18807
Name: count, dtype: int64
